# IBKR 1 分鐘歷史 K 回補 v1.5｜NQ、ES、TSM、SOXX（SOX/SDL 選用）

**Windows 本機 Jupyter + TWS API，唯讀不下單**。本 Notebook 必須和 `ibkr_history_core.py` 放在同一資料夾。保留期貨各季度原始合約，不把跨合約價差直接算成報酬。

**操作順序：** 安裝 → 連線 → SOX/SDL 選用測試 → 4 標的 smoke test → 手動啟動分批正式回補 → 驗證／Google Drive 同步。

**進度:** 每張合約顯示已完成的日曆日期窗口 / 尚待完成窗口、已寫入 K 棒、API 完成請求數、經過時間與 ETA。首輪可能要等待 IBKR 找到最早可用歷史時間。`paused_contract_slice` / `paused_budget` 是**正常暫停**，不是失敗；重新執行正式 Cell 接續進行。

**v1.5 修正：** 自動從正式下載計畫排除起始晚於終點的未來期貨合約；NQ、ES、TSM、SOXX 輪流分配下載額度；SOXX / TSM 最早歷史時間查詢失敗會快取 7 天，以免每次重複逾時。若使用 1990 年備援下界，下載到連續 10 個美國工作日無分鐘資料時會停止並寫 `unverified_history_boundary.json`（不宣稱找到真正的最早資料）。保留原本斷點續傳及 Parquet chunks。


**v1.5 安全修正：** 已到期期貨的計畫不再延伸到最後交易日後兩天；完整週末休市日自動略過且留下紀錄；單一合約發生無法判讀的空回應，會標示 `failed_retryable`、保留 checkpoint 並繼續其他商品。若 TWS 斷線或觸發 pacing，整批仍會停止。


**Repository import 2026-10-24:** Outputs and execution metadata removed. No automatic installation. DATA_ROOT defaults to a separate verified history copy. ALLOW_CONNECT=False and ALLOW_DOWNLOAD=False; all API/download cells require explicit opt-in. Cell 13 retains the original unlimited monitored-run settings: review budgets before enabling. The original notebook SHA256 is recorded in docs/v15_source_manifest.json.


## 1. 安裝套件

在原本已能連線 IBKR 的 Jupyter Kernel 執行；若安裝後無法匯入，重啟 Kernel。進度條需要 `tqdm` / `ipywidgets`。

In [ ]:
# Install requirements.txt in a separate environment before opening this notebook.
# No automatic package installation in the existing kernel.


## 2. Configuration

Select a verified snapshot created with `IBKR_HISTORY_COPY.ipynb` or `copy_existing_history.py`. The default is `~/Documents/IBKR_Data_Sandbox/IBKR_MarketData`; the original source dataset is not selected. Review the data path, client ID and request budgets before enabling ALLOW_CONNECT and ALLOW_DOWNLOAD. Final expiry sessions are excluded by default, not certified complete.


In [ ]:
from pathlib import Path
from ib_async import IB
from ibkr_history_core import (
    Settings, run_backfill, probe_indices, load_contract_data,
    contract_folder, validate_frame, gap_report,
    sync_tree_to_google_drive, reports_for_contract,
)
import pandas as pd

# 直接接續你先前成功下載的 16 個 Parquet / checkpoint。
# 如果已搬動資料夾，請修改這個路徑；避免不小心從空白目錄重新下載。
# Select a verified history COPY, never the original dataset.
DATA_ROOT = Path.home() / "Documents" / "IBKR_Data_Sandbox" / "IBKR_MarketData"
ALLOW_CONNECT = False
ALLOW_DOWNLOAD = False
DRIVE_LOCAL_FOLDER = None  # 範例：Path(r"G:\我的雲端硬碟\00Quant_Research\IBKR_MarketData")

cfg = Settings(
    symbols=("NQ", "ES"),
    stocks=("TSM", "SOXX"),
    indices=(),  # 可選：SOX / SDL，必須先確認分鐘 K 有資料
    host="127.0.0.1", port=7496, client_id=26,
    output_dir=str(DATA_ROOT),
    smoke_test=True,
    smoke_requests_per_contract=3,
    coverage_mode="roll_window",  # 原始季月合約含轉倉重疊，不是連續指數
    min_interval_seconds=12.0,
    requests_per_10_minutes=45,
    max_requests_per_run=150,
    max_requests_per_contract_per_run=5,  # 每批平均推進所有標的，不讓 ES 獨占額度
    google_drive_dir=None,
    show_progress=True,
    equity_head_required=False,  # Head Timestamp 無回覆則保守回溯，標記 UNVERIFIED
    equity_fallback_year=1990,  # head timestamp 查詢失敗時的備援來源下界
    equity_lookback_years=10,  # ★TSM/SOXX 限定近 10 年；即使 IBKR 回報 20 多年也不會超出
    expired_future_end_policy="pre_expiry_session",  # ★避開到期最後交易時段的歷史資料空回應
    stop_on_unverified_boundary=True,  # 若 Head 查不到，遇連續無資料工作日暫停
    equity_empty_weekdays_to_stop=10,
    equity_head_cache_days=7,
)
cfg.validate()
print("接續既有資料根目錄:", DATA_ROOT.resolve())
print("已選商品:", cfg.symbols + cfg.stocks + cfg.indices)
print("已有 Parquet 檔案:",len(list(DATA_ROOT.rglob("*.parquet"))))

## 3. 連接已啟用唯讀 API 的 TWS

若這個 Jupyter Kernel 已有 `ib` 且仍連線，會沿用它；否則使用預設 `7496` 與新 Client ID `23`。

In [ ]:
if ALLOW_CONNECT:
    if "ib" not in globals() or not ib.isConnected():
        ib = IB()
        await ib.connectAsync(cfg.host, cfg.port, clientId=cfg.client_id,
                              readonly=True, timeout=15)
    assert ib.isConnected(), "TWS 尚未連線；先登入 TWS 並確認 Socket API 7496"
    print("Connected:", ib.isConnected(), "| Server version:", ib.client.serverVersion())

## 4. SOX / SDL 指數分鐘 K 探測（選用）

IBKR 找到 `ConId` 不代表真的有分鐘 K。以下測試最近一個**已完成的美股交易日**，避免週末測試回傳 0 筆。結果須確認資料時段與價格；若 `rows > 0` 才考慮在設定加 `indices=("SOX",)`。指數價格通常沒有可比較的成交量。

In [ ]:
RUN_INDEX_PROBE = False
if RUN_INDEX_PROBE and ALLOW_DOWNLOAD:
    index_results = await probe_indices(ib)
    display(index_results)
else:
    print("SOX/SDL 暫不回補；改成 True 才測試。")

## 5. 首次小規模測試（建議先執行）

4 個標的各取最多 3 個 1 日窗口；期貨選最新季月合約。進度條即時顯示，下載後會寫到 `raw/<symbol>/<contract>_<conId>/chunks/`，及 `checkpoint.json`。

有 `timeout`、`permission`、`pacing` 錯誤會停止，不會假裝成功。
**v1.5:** 小測試僅檢查最近交易時段分鐘 K，**不查最早歷史時間**；此步不代表完成全歷史回補。


In [ ]:
if ALLOW_DOWNLOAD:
    cfg.smoke_test = True
    cfg.refresh_only = False
    smoke_plan, smoke_results = await run_backfill(ib, cfg)
    display(smoke_results[["symbol", "localSymbol", "status", "requests", "new_rows"]])

## 6. 正式歷史回補（手動開啟）

`RUN_FULL` 保持 `False` 不會開始。準備好後改 `True`。

- **每次最多 150 個完成的日窗口，每張合約最多 5 個**；四個標的輪流推進，中止後重新執行本 Cell 即續傳。
- 期貨依季度保留原始合約，`roll_window` 是連續研究時段附近的原始合約集合；不等於「每張合約從掛牌到到期全部分鐘資料」。需要後者時設 `coverage_mode="full_contract"`，但用新的資料目錄以避免 checkpoint 語義混淆。
- 股票從 `reqHeadTimeStampAsync(TRADES)` 回報的最早時間開始規劃，**不保證每個較舊年份都有分鐘 K**。
- `150 × 12 秒` 已約 30 分鐘，未含網路時間與報告處理。不要關閉 TWS 或讓 Windows 進入睡眠。
**v1.5：** `download_plan.csv` 包含 `head_source` / `head_error`，若顯示 `fallback_floor_UNVERIFIED`，代表 IBKR 未提供最早資料日期，1990 年只是回溯下界。遇連續 10 個美國工作日沒有分鐘資料時會記錄 `boundary_unverified_review`，請人工檢查，不可宣稱取得最長資料。`head_timestamp_cache.json` 避免重複無效 head 查詢。未來尚未開始的期貨窗口會記錄在 `skipped_future_start_plans.csv`，不列入正式工作進度。`completed_selected` 表示目前規劃範圍完成，**不代表歷史資料完整性已獲證實**。

- **v1.5：** `failed_retryable` 表示該合約有待確認的空回應／錯誤，**不能視為已補完**。失敗細節在 `failures.jsonl` 和 `run_summary.csv`；原合約 checkpoint 停在失敗日前，其他商品會繼續下載。`weekly_closed_skips` 是沒有呼叫 API 的已知週末休市日數。


- **v1.5：** 若 `ESM6`／`NQM6` 再遇到 2026-06-19 00:00 UTC 空回應，檢查是否錯誤載入舊版核心模組。預計新版這兩張合約 `end_utc` 為 **2026-06-17 21:00 UTC**；它刻意排除最後結算交易時段並留檔。股票/ETF 回溯起點應約為 **2016-10-18 UTC**（以 2026-10-18 執行為例）。


In [ ]:
RUN_FULL = False  # 確認四個標的 smoke test 正常後改 True
if RUN_FULL and ALLOW_DOWNLOAD:
    cfg.smoke_test = False
    cfg.refresh_only = False
    print("原有 Parquet:", len(list(DATA_ROOT.rglob("*.parquet"))))
    full_plan, full_results = await run_backfill(ib, cfg)
    cols = [c for c in ["symbol", "localSymbol", "status", "requests", "new_rows",
                       "weekly_closed_skips", "coverage_notice", "error"] if c in full_results.columns]
    display(full_results[cols])
    print("查看 excluded_expiry_terminal_windows.csv：到期最後時段有意排除，尚未驗證完整。")
    print("failed_retryable：該合約保留 checkpoint 待重試；其他標的照常執行。")
    print("boundary_unverified_review：尚未確認真正歷史起點，不是已完整回補。")
else:
    print("尚未啟動正式回補；改 RUN_FULL=True 後執行即可。")

In [ ]:
if ALLOW_DOWNLOAD:
    from ibkr_backfill_watchdog import (
        run_with_monitor,
        show_monitor_status
    )

    assert ib.isConnected(), "TWS 尚未連線"

    cfg.smoke_test = False
    cfg.refresh_only = False

    # 方案 A：持續下載，不限每輪請求數
    cfg.max_requests_per_contract_per_run = None
    cfg.max_requests_per_run = None

    # 保留 API 安全限速
    cfg.min_interval_seconds = 12.0
    cfg.requests_per_10_minutes = 45
    cfg.show_progress = True

    # 啟動連線監控與完整回補
    full_plan, full_results = await run_with_monitor(
        ib, cfg, run_backfill
    )

    display(full_results[[
        "symbol", "localSymbol",
        "status", "requests", "new_rows"
    ]])


### 回補安全診斷（選用）
顯示 `ESM6` 的新版回補終點及過往錯誤紀錄；此步不呼叫 IBKR。

v1.5 會額外建立 `excluded_expiry_terminal_windows.csv`；這不是因無成交而跳過，而是回補政策主動不處理到期最後的交易時段。


In [ ]:
# 查看最近一次規劃的 ESM6 時間邊界與失敗紀錄（不會發出 API 請求）
plan_path = DATA_ROOT / "download_plan.csv"
if plan_path.exists():
    last_plan = pd.read_csv(plan_path)
    display(last_plan.loc[last_plan["localSymbol"].isin(["ESM6", "NQM6"]) ,
                          ["symbol", "localSymbol", "start_utc", "end_utc"]])
else:
    print("尚未產生 download_plan.csv")
terminal_path = DATA_ROOT / "excluded_expiry_terminal_windows.csv"
if terminal_path.exists():
    print("特別注意：下列到期尾端時間尚未回補：")
    terminal = pd.read_csv(terminal_path)
    display(terminal.loc[terminal.localSymbol.isin(["ESM6", "NQM6"])])
fail_path = DATA_ROOT / "failures.jsonl"
if fail_path.exists():
    print("最近 3 次失敗紀錄（包含舊版歷史錯誤）：")
    print("\n".join(fail_path.read_text(encoding="utf-8").splitlines()[-3:]))


## 7. 顯示目前各標的的儲存進度

這裡依照 `chunks` 檔案數量統計已儲存窗口（只統計有資料的窗口，空白日期不產生 Parquet），不依賴 API；方便在下次打開 Notebook 時確認先前結果。

**v1.5：** 需要針對本次錯誤確認 `ESM6` 的新版計畫時，請在正式回補前執行下一格安全診斷。


In [ ]:
rows = []
for sym in (*cfg.symbols, *cfg.stocks, *cfg.indices):
    for folder in sorted((DATA_ROOT / "raw" / sym).glob("*")):
        if not folder.is_dir():
            continue
        chunks = list((folder / "chunks").glob("*.parquet"))
        rows.append({"symbol":sym, "contract":folder.name,
                     "saved_day_windows":len(chunks),
                     "saved_size_mb":round(sum(p.stat().st_size for p in chunks)/1024**2,2),
                     "checkpoint":(folder/"checkpoint.json").exists(),
                     "history_boundary_review":(folder/"unverified_history_boundary.json").exists()})
progress_table = pd.DataFrame(rows)
display(progress_table if len(progress_table) else "尚無資料")

## 8. 資料品質檢查

每個成功寫入的日窗口有 `reports/quality_by_chunk.jsonl`（重複、缺值、OHLC 邏輯等）；少於 120 個日窗口時會自動產生整張合約 `consolidated.parquet` 與 `reports/gaps.csv`。

當累積超過 120 個日窗口時，為避免每跑一次都讀取多年資料造成記憶體耗盡，會自動**停止產生 consolidated**；下列 Cell 可手動對指定合約執行完整檢查。缺口報告中的交易時間僅為一般週期的推估，未包含所有假日、提前休市、停牌／指數更新差異，不可把它直接等同資料供應商缺漏。

In [ ]:
RUN_FULL_AUDIT = False  # 大型股票資料可能需要大量 RAM，預設關閉
AUDIT_SYMBOL = "NQ"
if RUN_FULL_AUDIT:
    folders = sorted((DATA_ROOT / "raw" / AUDIT_SYMBOL).glob("*"))
    if not folders:
        raise ValueError(f"尚未下載 {AUDIT_SYMBOL}")
    folder = folders[-1]
    data = load_contract_data(folder)
    sec_type = "FUT" if AUDIT_SYMBOL in ("NQ", "ES") else ("IND" if AUDIT_SYMBOL in ("SOX","SDL") else "STK")
    quality = validate_frame(data)
    gaps = gap_report(data, sec_type=sec_type)
    audit_folder = folder / "reports"
    audit_folder.mkdir(exist_ok=True, parents=True)
    gaps.to_csv(audit_folder / "full_gaps.csv", index=False)
    print("Audited:", folder, quality)
    display(gaps.tail(20))
else:
    print("可針對某個合約開啟 RUN_FULL_AUDIT=True 執行完整缺口稽核。")

## 9. Google Drive for desktop 同步

先在 Windows 檔案總管確認「Google Drive for desktop」同步資料夾的**真正本機路徑**；將第 2 節的 `DRIVE_LOCAL_FOLDER` 改為實際路徑後執行。這只表示資料已複製到同步資料夾，**不是 Google 雲端上傳完成證明**，上傳完成需要查看 Google Drive 桌面程式。

若希望下次每跑完一張合約切片就同步，成功設定後再執行完整回補；每次只複製修改過的檔案。不包含券商帳號、憑證或下單功能。

In [ ]:
if DRIVE_LOCAL_FOLDER is None:
    print("尚未設定 Google Drive for desktop 的本機同步路徑；資料只在本機。")
else:
    DRIVE_LOCAL_FOLDER = Path(DRIVE_LOCAL_FOLDER)
    if not DRIVE_LOCAL_FOLDER.is_dir():
        raise FileNotFoundError(f"同步目的資料夾不存在：{DRIVE_LOCAL_FOLDER}")
    print(sync_tree_to_google_drive(DATA_ROOT, DRIVE_LOCAL_FOLDER))
    cfg.google_drive_dir = str(DRIVE_LOCAL_FOLDER)

## 10. 後續增量更新（選用）

先確認正式回補沒有待補的缺口。`refresh_days=7` 只重抓最近 7 天；若停機更久需適度加大；換季仍須確認季度合約更新及轉倉邊界。增量更新也遵守 pacing 和每次工作額度。

In [ ]:
RUN_REFRESH = False
if RUN_REFRESH and ALLOW_DOWNLOAD:
    cfg.smoke_test = False
    cfg.refresh_only = True
    cfg.refresh_days = 7
    _, refresh_results = await run_backfill(ib, cfg)
    display(refresh_results[["symbol", "status", "requests", "new_rows"]])
    cfg.refresh_only = False
else:
    print("增量更新目前關閉。")

## 11. 研究注意事項

- **期貨** `NQ/ES`：只有同一合約的價格能直接運算報酬；換月轉倉規則、主力判定、回測資訊時點需要另外設計，不能直接串接價格。
- **股票** `TSM/SOXX`：`useRTH=False` 包含可得的盤前／盤後成交，不代表全天 24 小時有 K 棒；歷史分鐘資料可能比最早的歷史 timestamp 少。預設回傳的原始價格不是企業行動全調整序列，研究跨年報酬須處理拆股／股利與版本紀錄。
- **指數** `SOX/SDL`：`ConId` 成功不代表分鐘 K 可回補；請先測試，再手動納入一種合適的指數合約，不要無條件把 SOX 與 SDL 混成同一條價格序列。
- **時間** 全部儲存 UTC，與台灣現貨對齊時使用 `Asia/Taipei`；美國夏令時間判斷用 `America/New_York`／`America/Chicago`。
- **資料權利** Google Drive 僅建議私人授權範圍保存，不公開再散布交易所市場資料。

官方參考：[Historical Bars](https://www.interactivebrokers.com/docs/tws-api/doc/md/historical-bars)；[Historical Limitations](https://interactivebrokers.github.io/tws-api/historical_limitations.html)。

- **v1.5 特別標示：** `pre_expiry_session` 到期政策會主動省略每個已到期期貨合約的最後結算時段；這些分鐘**不是已證明沒有交易或完整取得**。實際研究請依 `excluded_expiry_terminal_windows.csv` 與新合約資料核對。


In [ ]:
root = Path(cfg.output_dir).resolve()

for file in root.rglob("*"):
    if file.is_file():
        print(file.relative_to(root))